# Granite 4.1 — Tool Calling

## Imports

In [1]:
import json
import re
from pprint import pprint

import torch
import transformers

print("torch:", torch.__version__)
print("MPS (Apple Silicon GPU) available:", torch.backends.mps.is_available())
print("CUDA available:", torch.cuda.is_available())

torch: 2.13.0
MPS (Apple Silicon GPU) available: True
CUDA available: False


## Load Model and Tokenizer

In [2]:
MODEL_ID = "ibm-granite/granite-4.1-3b"

tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_ID)
model = transformers.AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype="auto",
    device_map="auto",
)

print(f"Architecture: {model.config.architectures}")
print(f"Parameters: {model.num_parameters():,}")
print(f"Device: {model.device}")
print(f"Dtype: {model.dtype}")

Loading weights:   0%|          | 0/362 [00:00<?, ?it/s]

Architecture: ['GraniteForCausalLM']
Parameters: 3,402,836,480
Device: mps:0
Dtype: torch.bfloat16


## Define Tools

In [3]:
def get_weather(location: str) -> str:
    """Get the current weather for a location.

    Args:
        location: The city and state, e.g. San Francisco, CA
    """
    return json.dumps({
        "location": location,
        "temperature": "68",
        "unit": "fahrenheit",
        "condition": "sunny",
    })


def get_stock_price(symbol: str) -> str:
    """Get the current stock price for a ticker symbol.

    Args:
        symbol: The stock ticker symbol, e.g. AAPL
    """
    return json.dumps({
        "symbol": symbol,
        "price": 294.38,
        "currency": "USD",
    })


def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount of money from one currency to another.

    Args:
        amount: The amount of money to convert
        from_currency: The three-letter currency code to convert from, e.g. USD
        to_currency: The three-letter currency code to convert to, e.g. EUR
    """
    rate = 0.88
    return json.dumps({
        "converted_amount": round(amount * rate, 2),
        "to_currency": to_currency},
    )


AVAILABLE_TOOLS = {
    "get_weather": get_weather,
    "get_stock_price": get_stock_price,
    "convert_currency": convert_currency,
}

tools = [transformers.utils.get_json_schema(fn) for fn in AVAILABLE_TOOLS.values()]

pprint(tools, sort_dicts=False, width=120)

[{'type': 'function',
  'function': {'name': 'get_weather',
               'description': 'Get the current weather for a location.',
               'parameters': {'type': 'object',
                              'properties': {'location': {'type': 'string',
                                                          'description': 'The city and state, e.g. San Francisco, CA'}},
                              'required': ['location']},
               'return': {'type': 'string'}}},
 {'type': 'function',
  'function': {'name': 'get_stock_price',
               'description': 'Get the current stock price for a ticker symbol.',
               'parameters': {'type': 'object',
                              'properties': {'symbol': {'type': 'string',
                                                        'description': 'The stock ticker symbol, e.g. AAPL'}},
                              'required': ['symbol']},
               'return': {'type': 'string'}}},
 {'type': 'function',
  'function': {

## Single Tool Call

In [4]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco?"},
]

chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currenc

In [5]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

<tool_call>
{"name": "get_weather", "arguments": {"location": "San Francisco, CA"}}
</tool_call>


In [6]:
tool_calls = [
    json.loads(match)
    for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
]
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}}]


In [7]:
call = tool_calls[0]
result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
print(result)

{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}


In [8]:
messages.append({"role": "assistant", "tool_calls": tool_calls})
messages.append({"role": "tool", "content": result})

pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the weather in San Francisco?"},
 {'role': 'assistant', 'tool_calls': [{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}}]},
 {'role': 'tool',
  'content': '{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}'}]


In [9]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currenc

In [10]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The current weather in San Francisco is sunny with a temperature of 68°F.


## Multiple Tool Calls

In [11]:
messages = [
    {"role": "user", "content": "What's the weather in San Francisco and the stock price of AAPL?"},
]

inputs = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=True,
    return_tensors="pt",
    add_generation_prompt=True,
).to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

<tool_call>
{"name": "get_weather", "arguments": {"location": "San Francisco, CA"}}
</tool_call>
<tool_call>
{"name": "get_stock_price", "arguments": {"symbol": "AAPL"}}
</tool_call>


In [12]:
tool_calls = [
    json.loads(match)
    for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
]
pprint(tool_calls, sort_dicts=False, width=120)

[{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}},
 {'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]


In [13]:
messages.append({"role": "assistant", "tool_calls": tool_calls})
for call in tool_calls:
    result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
    messages.append({"role": "tool", "content": result})

pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What's the weather in San Francisco and the stock price of AAPL?"},
 {'role': 'assistant',
  'tool_calls': [{'name': 'get_weather', 'arguments': {'location': 'San Francisco, CA'}},
                 {'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]},
 {'role': 'tool',
  'content': '{"location": "San Francisco, CA", "temperature": "68", "unit": "fahrenheit", "condition": "sunny"}'},
 {'role': 'tool', 'content': '{"symbol": "AAPL", "price": 294.38, "currency": "USD"}'}]


In [14]:
chat = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=True,
)

print(chat)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currenc

In [15]:
inputs = tokenizer(chat, return_tensors="pt").to(model.device)
input_len = inputs["input_ids"].shape[-1]

outputs = model.generate(**inputs, max_new_tokens=128)
response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

print(response)

The current weather in San Francisco is sunny with a temperature of 68°F. The stock price for AAPL is $294.38 USD.


## Sequential Tool Calls

In [16]:
messages = [
    {"role": "user", "content": "What is AAPL's stock price converted to EUR?"},
]

print(f"[user]\n{messages[0]['content']}\n")

while True:
    inputs = tokenizer.apply_chat_template(
        messages,
        tools=tools,
        tokenize=True,
        return_tensors="pt",
        add_generation_prompt=True,
    ).to(model.device)
    input_len = inputs["input_ids"].shape[-1]

    outputs = model.generate(**inputs, max_new_tokens=128)
    response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)

    tool_calls = [
        json.loads(match)
        for match in re.findall(r"<tool_call>\s*(.*?)\s*</tool_call>", response, re.DOTALL)
    ]
    if not tool_calls:
        break

    messages.append({"role": "assistant", "tool_calls": tool_calls})
    for call in tool_calls:
        result = AVAILABLE_TOOLS[call["name"]](**call["arguments"])
        print(f"[tool call]\n{call['name']}({call['arguments']})\n")
        print(f"[tool response]\n{result}\n")
        messages.append({"role": "tool", "content": result})

print("[assistant]")
print(response)

[user]
What is AAPL's stock price converted to EUR?

[tool call]
get_stock_price({'symbol': 'AAPL'})

[tool response]
{"symbol": "AAPL", "price": 294.38, "currency": "USD"}

[tool call]
convert_currency({'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'})

[tool response]
{"converted_amount": 259.05, "to_currency": "EUR"}

[assistant]
AAPL’s current stock price is **$294.38 USD**, which converts to **≈ €259.05 EUR**.


In [17]:
messages.append({"role": "assistant", "content": response})
pprint(messages, sort_dicts=False, width=120)

[{'role': 'user', 'content': "What is AAPL's stock price converted to EUR?"},
 {'role': 'assistant', 'tool_calls': [{'name': 'get_stock_price', 'arguments': {'symbol': 'AAPL'}}]},
 {'role': 'tool', 'content': '{"symbol": "AAPL", "price": 294.38, "currency": "USD"}'},
 {'role': 'assistant',
  'tool_calls': [{'name': 'convert_currency',
                  'arguments': {'amount': 294.38, 'from_currency': 'USD', 'to_currency': 'EUR'}}]},
 {'role': 'tool', 'content': '{"converted_amount": 259.05, "to_currency": "EUR"}'},
 {'role': 'assistant',
  'content': 'AAPL’s current stock price is **$294.38 USD**, which converts to **≈\u202f€259.05 EUR**.'}]


In [18]:
conversation = tokenizer.apply_chat_template(
    messages,
    tools=tools,
    tokenize=False,
    add_generation_prompt=False,
)

print(conversation)

<|start_of_role|>system<|end_of_role|>You are a helpful assistant with access to the following tools. You may call one or more tools to assist with the user query.

You are provided with function signatures within <tools></tools> XML tags:
<tools>
{"type": "function", "function": {"name": "get_weather", "description": "Get the current weather for a location.", "parameters": {"type": "object", "properties": {"location": {"type": "string", "description": "The city and state, e.g. San Francisco, CA"}}, "required": ["location"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "get_stock_price", "description": "Get the current stock price for a ticker symbol.", "parameters": {"type": "object", "properties": {"symbol": {"type": "string", "description": "The stock ticker symbol, e.g. AAPL"}}, "required": ["symbol"]}, "return": {"type": "string"}}}
{"type": "function", "function": {"name": "convert_currency", "description": "Convert an amount of money from one currenc